# Incremental Dataset Sampling for Supervised Learning

## Objective

In this experiment, we want to investigate how the size of the training dataset affects the performance of a supervised learning model.

Instead of training the model on the entire dataset from the beginning, we gradually increase the training data size in batches of 100,000 samples.

The experiments will use the following training sizes:

- 100,000 samples
- 200,000 samples
- 300,000 samples
- 400,000 samples
- And so on, until the desired performance is achieved.

The goal is to identify a suitable training dataset size without unnecessarily using the entire dataset.

---

## 1. Creating a Fixed Validation Set

First, we create a separate validation set containing 50,000 labeled samples.

The validation set is sampled with approximately balanced classes to ensure that both recommendation categories are sufficiently represented during model evaluation.

**Important points:**

- The validation set is created only once.
- Its samples are excluded from all training batches.
- The same validation set is used in every experiment.
- The validation set is not used for model training.

Keeping the validation set fixed allows us to compare model performance across different training dataset sizes more reliably.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# File paths
SOURCE_PATH = Path("../data/digikala-comments-final.csv")
DATA_DIR = Path("../data")

# Dataset settings
ID_COL = "id"
TARGET_COL = "recommendation_status"
RATING_COL = "rate"
TEXT_COL = "text"
RANDOM_SEED = 42

BATCH_SIZE = 100_000


In [2]:
def split_by_product_id(
    source_path,
    sample_size,
    excluded_product_ids=None,
    seed=42,
    labels=None,
    chunk_size=100_000
):
    """
    Create a balanced sample while keeping product groups separate.
    """

    if excluded_product_ids is None:
        excluded_product_ids = set()

    if labels is None:
        labels = [
            "recommended",
            "not_recommended",
            "no_idea"
        ]

    # Calculate the quota for each class
    base_quota = sample_size // len(labels)
    remainder = sample_size % len(labels)

    quotas = {
        label: base_quota + (i < remainder)
        for i, label in enumerate(labels)
    }

    # Detect the text column
    source_columns = pd.read_csv(
        source_path,
        nrows=0
    ).columns.tolist()

    text_source_col = (
        "text" if "text" in source_columns else "body_normalized"
    )

    required_columns = [
        ID_COL,
        "product_id",
        text_source_col,
        TARGET_COL,
        RATING_COL
    ]

    missing_columns = [
        col for col in required_columns
        if col not in source_columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns: {missing_columns}"
        )

    # --------------------------------------------------
    # Pass 1: Count comments per product and class
    # --------------------------------------------------

    product_count_parts = []
    seen_ids = set()

    for chunk in pd.read_csv(
        source_path,
        usecols=required_columns,
        chunksize=chunk_size,
        dtype=str,
        keep_default_na=False
    ):

        chunk[ID_COL] = chunk[ID_COL].str.strip()
        chunk["product_id"] = chunk["product_id"].str.strip()

        chunk[TARGET_COL] = (
            chunk[TARGET_COL].str.strip().str.lower()
        )

        # Keep valid comments and available products
        chunk = chunk[
            chunk[ID_COL].ne("")
            & chunk["product_id"].ne("")
            & chunk[text_source_col].str.strip().ne("")
            & chunk[TARGET_COL].isin(labels)
            & ~chunk["product_id"].isin(excluded_product_ids)
        ].copy()

        # Remove duplicate comment IDs
        chunk = chunk[
            ~chunk[ID_COL].isin(seen_ids)
            & ~chunk[ID_COL].duplicated()
        ]

        seen_ids.update(chunk[ID_COL])

        if chunk.empty:
            continue

        # Count comments of each class for every product
        counts = (
            chunk.groupby(["product_id", TARGET_COL])
            .size()
            .unstack(fill_value=0)
        )

        product_count_parts.append(counts)

    if not product_count_parts:
        raise ValueError("No available comments found.")

    product_stats = (
        pd.concat(product_count_parts)
        .groupby(level=0)
        .sum()
        .reindex(columns=labels, fill_value=0)
    )

    # Check whether enough comments are available
    available_counts = product_stats.sum()

    print("Available comments:")
    print(available_counts)

    for label in labels:
        if available_counts[label] < quotas[label]:
            raise ValueError(
                f"Not enough comments for {label}. "
                f"Available: {available_counts[label]}, "
                f"Required: {quotas[label]}"
            )

    # --------------------------------------------------
    # Pass 2: Select product groups
    # --------------------------------------------------

    quota_series = pd.Series(quotas)

    product_scores = (
    product_stats.clip(upper=quota_series, axis=1)
    .div(quota_series, axis=1)
    .sum(axis=1)
    )
    rng = np.random.default_rng(seed)

    ranked_products = product_stats.copy()

    ranked_products["_score"] = product_scores
    ranked_products["_random"] = rng.random(
        len(ranked_products)
    )

    ranked_products = ranked_products.sort_values(
        ["_score", "_random"],
        ascending=False
    )

    selected_product_ids = set()

    selected_counts = {
        label: 0 for label in labels
    }

    for product_id, row in ranked_products.iterrows():

        # Stop when all class quotas are satisfied
        if all(
            selected_counts[label] >= quotas[label]
            for label in labels
        ):
            break

        # Check whether this product contributes to a needed class
        contributes = any(
            row[label] > 0
            and selected_counts[label] < quotas[label]
            for label in labels
        )

        if not contributes:
            continue

        selected_product_ids.add(product_id)

        for label in labels:
            remaining = quotas[label] - selected_counts[label]

            selected_counts[label] += min(
                int(row[label]),
                remaining
            )

    for label in labels:
        if selected_counts[label] < quotas[label]:
            raise RuntimeError(
                f"Could not select enough products for {label}."
            )

    # --------------------------------------------------
    # Pass 3: Collect comments from selected products
    # --------------------------------------------------

    selected_parts = []
    seen_sample_ids = set()

    for chunk in pd.read_csv(
        source_path,
        usecols=required_columns,
        chunksize=chunk_size,
        dtype=str,
        keep_default_na=False
    ):

        chunk[ID_COL] = chunk[ID_COL].str.strip()
        chunk["product_id"] = chunk["product_id"].str.strip()

        chunk[TARGET_COL] = (
            chunk[TARGET_COL].str.strip().str.lower()
        )

        chunk = chunk[
            chunk["product_id"].isin(selected_product_ids)
            & chunk[TARGET_COL].isin(labels)
            & chunk[ID_COL].ne("")
            & chunk[text_source_col].str.strip().ne("")
        ].copy()

        chunk = chunk[
            ~chunk[ID_COL].isin(seen_sample_ids)
            & ~chunk[ID_COL].duplicated()
        ]

        seen_sample_ids.update(chunk[ID_COL])

        if not chunk.empty:
            selected_parts.append(chunk)

    if not selected_parts:
        raise RuntimeError("No comments found for selected products.")

    candidate_df = pd.concat(
        selected_parts,
        ignore_index=True
    )

    # --------------------------------------------------
    # Pass 4: Select the exact quota for each class
    # --------------------------------------------------

    sample_parts = []

    for i, label in enumerate(labels):

        class_df = candidate_df[
            candidate_df[TARGET_COL] == label
        ]

        if len(class_df) < quotas[label]:
            raise RuntimeError(
                f"Not enough candidates for {label}."
            )

        class_sample = class_df.sample(
            n=quotas[label],
            random_state=seed + i
        )

        sample_parts.append(class_sample)

    sample_df = pd.concat(
        sample_parts,
        ignore_index=True
    )

    sample_df = sample_df.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    # Keep required columns
    sample_df = sample_df[
        [
            ID_COL,
            "product_id",
            text_source_col,
            TARGET_COL,
            RATING_COL
        ]
    ].rename(
        columns={text_source_col: TEXT_COL}
    )

    # Convert rating values to numeric
    sample_df[RATING_COL] = pd.to_numeric(
        sample_df[RATING_COL],
        errors="coerce"
    )

    return sample_df, selected_product_ids

In [5]:
# Define validation settings
VALIDATION_SIZE = 50_000

LABELS = [
    "recommended",
    "not_recommended",
    "no_idea"
]



In [ ]:
# No products are reserved at the beginning
reserved_product_ids = set()

# Create validation dataset
validation_df, validation_product_ids = split_by_product_id(
    source_path=SOURCE_PATH,
    sample_size=VALIDATION_SIZE,
    excluded_product_ids=reserved_product_ids,
    seed=RANDOM_SEED,
    labels=LABELS,
    chunk_size=VALIDATION_SIZE
)

# Reserve selected validation products
reserved_product_ids.update(validation_product_ids)

# Save validation dataset
VALIDATION_PATH = DATA_DIR / "validation_50k.csv"

validation_df.to_csv(
    VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Save reserved product IDs
RESERVED_PRODUCTS_PATH = DATA_DIR / "reserved_product_ids.csv"

pd.DataFrame({
    "product_id": sorted(reserved_product_ids)
}).to_csv(
    RESERVED_PRODUCTS_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Display results
print("Validation shape:", validation_df.shape)

print("\nClass distribution:")
print(
    validation_df[TARGET_COL]
    .value_counts()
    .reindex(LABELS, fill_value=0)
)

print("\nUnique products:", validation_df["product_id"].nunique())
print("Reserved products:", len(reserved_product_ids))

print("\nSaved:", VALIDATION_PATH.name)

Available comments:
recommendation_status
recommended        3200264
not_recommended     452496
no_idea             467660
dtype: int64
Validation shape: (50000, 5)

Class distribution:
recommendation_status
recommended        16667
not_recommended    16667
no_idea            16666
Name: count, dtype: int64

Unique products: 999
Reserved products: 999

Saved: validation_50k.csv


In [ ]:
df = pd.read_csv()

## 2. Creating Training Batches

After creating the validation set, we generate training batches.

Each batch contains 100,000 labeled samples. Samples are drawn with balanced classes: the quota is split equally across `recommended`, `not_recommended` and `no_idea` (33,334 / 33,333 / 33,333, with the remainder going to the first classes).

To prevent data leakage between batches and the validation set, sampling is done at the **product level**:

- Products used in the validation set are excluded from every training batch.
- Products used in previously generated batches are excluded from the next ones, so batches are product-disjoint from each other.
- Comments with a missing `id`, `product_id`, text or label (or a label outside the three classes) are skipped, and duplicate comment IDs are dropped, so each sample has a unique ID.
- The selected `product_id` values are appended to `reserved_product_ids.csv` after each batch is created. This file is the record of all reserved products (validation + batches).

Within the eligible products, the sampler picks products greedily so that the class quotas are filled, then draws exactly the required number of comments per class and shuffles the result.

The original dataset remains unchanged. Instead of deleting selected rows from the source file, we use `reserved_product_ids.csv` to keep track of previously selected products.

### Reproducibility

A fixed random seed is used during sampling (`RANDOM_SEED + BATCH_NUMBER`, so each batch has its own seed).

This helps make the sampling process reproducible, provided that the source file and its row order remain unchanged, and that batches are generated in the same order starting from the same `reserved_product_ids.csv` state (validation set first, then batch 1, 2, ...).

In [134]:
# Create a training batch

CHUNK_SIZE = 100_000
BATCH_NUMBER = 10

BATCH_PATH = DATA_DIR / (
    f"train_batch_{BATCH_NUMBER:02d}.csv"
)

if BATCH_PATH.exists():
    raise FileExistsError(
        f"{BATCH_PATH.name} already exists. "
        "Choose a different batch number."
    )

# --------------------------------------------------
# Load previously reserved product IDs
# --------------------------------------------------

if RESERVED_PRODUCTS_PATH.exists():

    reserved_product_ids = set(
        pd.read_csv(
            RESERVED_PRODUCTS_PATH,
            dtype=str
        )["product_id"].str.strip()
    )

else:
    raise FileNotFoundError(
        "Reserved product IDs file not found. "
        "Create Validation first."
    )

print("Previously reserved products:", len(reserved_product_ids))

# --------------------------------------------------
# Create a new training batch
# --------------------------------------------------

batch_df, batch_product_ids = split_by_product_id(
    source_path=SOURCE_PATH,
    sample_size=BATCH_SIZE,
    excluded_product_ids=reserved_product_ids,
    seed=RANDOM_SEED + BATCH_NUMBER,
    labels=LABELS,
    chunk_size=CHUNK_SIZE
)

# --------------------------------------------------
# Verify the new batch
# --------------------------------------------------

assert len(batch_df) == BATCH_SIZE
assert batch_df[ID_COL].is_unique

# Verify product separation
assert batch_product_ids.isdisjoint(
    reserved_product_ids
)

# Get actual product IDs from the final batch
actual_product_ids = set(
    batch_df["product_id"].astype(str).str.strip()
)

# Normalize selected product IDs
selected_product_ids = set(
    str(pid).strip() for pid in batch_product_ids
)

# Compare the two sets
missing_from_batch = selected_product_ids - actual_product_ids
unexpected_in_batch = actual_product_ids - selected_product_ids

print("Selected product IDs:", len(selected_product_ids))
print("Actual batch product IDs:", len(actual_product_ids))

print("Selected products not present in batch:", len(missing_from_batch))
print("Unexpected products in batch:", len(unexpected_in_batch))

# Verify that every product in the batch was selected
assert actual_product_ids.issubset(selected_product_ids)

# --------------------------------------------------
# Save the new batch
# --------------------------------------------------

batch_df.to_csv(
    BATCH_PATH,
    index=False,
    encoding="utf-8-sig"
)

# --------------------------------------------------
# Update reserved product IDs
# --------------------------------------------------

reserved_product_ids.update(batch_product_ids)

pd.DataFrame({
    "product_id": sorted(reserved_product_ids)
}).to_csv(
    RESERVED_PRODUCTS_PATH,
    index=False,
    encoding="utf-8-sig"
)

# --------------------------------------------------
# Show results
# --------------------------------------------------

print(f"\nBatch {BATCH_NUMBER} created successfully!")

print("\nBatch shape:", batch_df.shape)

print("\nClass distribution:")
print(
    batch_df[TARGET_COL]
    .value_counts()
    .reindex(LABELS, fill_value=0)
)

print("\nUnique comment IDs:", batch_df[ID_COL].nunique())

print("Unique product IDs:", batch_df["product_id"].nunique())

print(
    "\nNewly reserved products:",
    len(batch_product_ids)
)

print(
    "Total reserved products:",
    len(reserved_product_ids)
)

print("\nSaved file:", BATCH_PATH.name)

Previously reserved products: 44866
Available comments:
recommendation_status
recommended        779170
not_recommended    135770
no_idea            126302
dtype: int64
Selected product IDs: 16600
Actual batch product IDs: 16591
Selected products not present in batch: 9
Unexpected products in batch: 0

Batch 10 created successfully!

Batch shape: (100000, 5)

Class distribution:
recommendation_status
recommended        33334
not_recommended    33333
no_idea            33333
Name: count, dtype: int64

Unique comment IDs: 100000
Unique product IDs: 16591

Newly reserved products: 16600
Total reserved products: 61466

Saved file: train_batch_10.csv


## 3. Building Training Sets for Experiments

Once the training batches have been generated, we can combine them to create training datasets of different sizes.

For example:

| Experiment | Training batches | Training size |
|---|---:|---:|
| 1 | 1 | 100,000 |
| 2 | 2 | 200,000 |
| 3 | 3 | 300,000 |
| 4 | 4 | 400,000 |

For each experiment, all batches from the beginning up to the selected batch number are combined.

**Important:** The model must be trained from scratch for each experiment. We should not train the previous model only on the newly added batch, because that would make the experiments less comparable.



In [135]:
from pathlib import Path
import pandas as pd

# Choose the current experiment
BATCH_COUNT = 10

# Create paths for the selected training batches
batch_paths = [
    DATA_DIR / f"train_batch_{i:02d}.csv"
    for i in range(1, BATCH_COUNT + 1)
]

# Check whether all batch files exist
for path in batch_paths:
    if not path.exists():
        raise FileNotFoundError(f"Batch file not found: {path}")

# Load and combine all selected batches
train_df = pd.concat(
    [
        pd.read_csv(path, dtype=str, keep_default_na=False)
        for path in batch_paths
    ],
    ignore_index=True
)

# Check for duplicate comment IDs
assert train_df[ID_COL].is_unique, "Duplicate comment IDs found!"

# Load Validation product IDs
validation_df = pd.read_csv(
    VALIDATION_PATH,
    usecols=["product_id"],
    dtype=str,
    keep_default_na=False
)

validation_product_ids = set(
    validation_df["product_id"].str.strip()
)

# Check for product overlap between Train and Validation
train_product_ids = set(
    train_df["product_id"].str.strip()
)

overlap = train_product_ids.intersection(validation_product_ids)

assert not overlap, (
    f"Product leakage detected: "
    f"{len(overlap)} products appear in both Train and Validation."
)

print("No product overlap between Train and Validation.")

# Save the training dataset for this experiment
train_path = DATA_DIR / f"train_experiment_{BATCH_COUNT * 100}k.csv"

train_df.to_csv(
    train_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nTraining dataset:", train_path.name)
print("Shape:", train_df.shape)

print("\nClass distribution:")
print(train_df[TARGET_COL].value_counts())

No product overlap between Train and Validation.

Training dataset: train_experiment_1000k.csv
Shape: (1000000, 5)

Class distribution:
recommendation_status
recommended        333340
no_idea            333330
not_recommended    333330
Name: count, dtype: int64


## 4. Experimental Evaluation

For each training dataset size:

1. Train a new model using the corresponding training set.
2. Evaluate the model on the same validation set.
3. Record the evaluation metrics.
4. Compare the results with previous experiments.

Useful evaluation metrics include:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix
- ROC-AUC
- PR-AUC

Since the training samples are approximately balanced, but the real-world dataset may have a different class distribution, evaluation should also be performed on data that reflects the natural class distribution.

The validation set should remain unchanged throughout these experiments.

---


### Modeling 

#### Imports

In [11]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT / "src"))

from tfidf_baseline import (
    load_data,
    create_rating_band,
    prepare_data,
    train_all_models
)

In [136]:
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

# Create results directory if it does not exist
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Define dataset paths
TRAIN_PATH = DATA_DIR / "train_experiment_1000k.csv"
VALIDATION_PATH = DATA_DIR / "validation_50k_corrected.csv"

# Load training and validation datasets
train_df, val_df = load_data(
    TRAIN_PATH,
    VALIDATION_PATH
)


Training shape: (1000000, 5)
Validation shape: (50000, 5)


In [137]:
# Create rating bands
train_df = create_rating_band(train_df)
val_df = create_rating_band(val_df)

# Prepare data for TF-IDF models
train_df = prepare_data(train_df)
val_df = prepare_data(val_df)

# Display dataset information
print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nTraining class distribution:")
print(train_df["recommendation_status"].value_counts())

print("\nValidation class distribution:")
print(val_df["recommendation_status"].value_counts())

Training shape: (1000000, 6)
Validation shape: (50000, 6)

Training class distribution:
recommendation_status
recommended        333340
no_idea            333330
not_recommended    333330
Name: count, dtype: int64

Validation class distribution:
recommendation_status
not_recommended    17391
recommended        17123
no_idea            15486
Name: count, dtype: int64


In [139]:
results = train_all_models(
    train_df=train_df,
    val_df=val_df
)


Training model for: recommendation_status

Target: recommendation_status
Training samples: 1,000,000
Validation samples: 50,000

Classification Report:
                 precision    recall  f1-score   support

        no_idea       0.63      0.68      0.65     15486
not_recommended       0.80      0.78      0.79     17391
    recommended       0.85      0.82      0.84     17123

       accuracy                           0.76     50000
      macro avg       0.76      0.76      0.76     50000
   weighted avg       0.77      0.76      0.76     50000

Confusion Matrix:
[[10454  2877  2155]
 [ 3544 13480   367]
 [ 2606   411 14106]]

Training model for: rating_band

Target: rating_band
Training samples: 1,000,000
Validation samples: 50,000

Classification Report:
              precision    recall  f1-score   support

        high       0.63      0.74      0.68     14506
         low       0.73      0.63      0.68     19509
      medium       0.54      0.54      0.54     15985

    accuracy

In [140]:
# Convert model results into a DataFrame
results_df = pd.DataFrame([
    {
        "target": target,
        **result["metrics"]
    }
    for target, result in results.items()
])
results_df


,target,actual_train_size,validation_size,accuracy,macro_f1,balanced_accuracy
0,recommendation_status,1000000,50000,0.76080,0.758893,0.757993
1,rating_band,1000000,50000,0.63404,0.632354,0.637683


In [141]:
output_path = RESULTS_DIR / "tfidf_baseline_results.csv"

results_df.to_csv(
    output_path,
    mode="a",
    header=not output_path.exists(),
    index=False
)

print("New results appended to:", output_path)

New results appended to: d:\AI Bootcamp_M Sharif\digikala_reviews_project\results\tfidf_baseline_results.csv


In [142]:
import pandas as pd
result = pd.read_csv("D:\\AI Bootcamp_M Sharif\\digikala_reviews_project\\results\\tfidf_baseline_results.csv")
display(result)

,target,actual_train_size,validation_size,accuracy,macro_f1,balanced_accuracy
0,recommendation_status,100000,50000,0.74614,0.744313,0.743309
1,rating_band,100000,50000,0.62156,0.619386,0.623279
2,recommendation_status,200000,50000,0.75212,0.750218,0.749240
3,rating_band,200000,50000,0.62422,0.621855,0.626646
4,recommendation_status,300000,50000,0.75420,0.752179,0.751263
5,rating_band,300000,50000,0.62602,0.623584,0.628751
6,recommendation_status,400000,50000,0.75512,0.753049,0.752156
7,rating_band,400000,50000,0.62848,0.626185,0.631409
8,recommendation_status,500000,50000,0.75682,0.754782,0.753892
9,rating_band,500000,50000,0.62968,0.627524,0.632393


### Conclusion: Impact of Training Dataset Size

To investigate the effect of training dataset size on classification performance, TF-IDF models with Logistic Regression were trained using different training sizes, ranging from 100,000 to 1,000,000 samples. The validation dataset remained fixed at 50,000 samples throughout all experiments.

The results demonstrate a consistent improvement in both classification tasks as the training dataset size increases.

For **Recommendation Status Classification**, Macro F1 increased from 0.7443 with 100K training samples to 0.7589 with 1M samples. Similarly, for **Rating Band Classification**, Macro F1 improved from 0.6194 to 0.6324.

Although the 1M training dataset achieved the highest performance, doubling the training data from 500K to 1M resulted in relatively small improvements in both tasks. This suggests diminishing returns from increasing the training dataset size.

Based on these results, **500K samples were selected as the primary training dataset for subsequent experiments**. This size provides a reasonable balance between classification performance and computational efficiency, particularly considering the higher computational cost of Transformer fine-tuning.

The 1M dataset will remain available for additional experiments to investigate whether larger training datasets provide further improvements in Transformer-based classification.

Overall, these experiments establish a TF-IDF baseline and provide a reference point for evaluating the effectiveness of the Transformer model.


## 5. Creating the Test Set (10,000 comments)

After the training batches and the validation set are final, we create a fixed test set of 10,000 labeled comments.

**Guarantees (all verified explicitly at the end):**

- **Product separation:** no product in the test set appears in any training batch or in the validation set (original or corrected). Reserved products from `reserved_product_ids.csv` are excluded as well.
- **Body separation:** no body in the test set matches any training or validation body. Bodies are compared after normalization (Arabic/Persian characters and digits unified, punctuation, emojis, diacritics, ZWNJ and spaces removed, repeated letters collapsed), so variants such as `عالی`, `عالی!!!`, `عالييي` and `عالی ` count as the same body. A second, order-insensitive signature (sorted set of words) also catches re-ordered bodies.
- **No duplicates inside the test set:** each normalized body appears once, and bodies whose character n-gram cosine similarity is at least `NEAR_DUP_THRESHOLD` are removed as near-duplicates.
- **Clean labels:** a body that appears with different labels anywhere in the candidate pool is removed entirely (ambiguous label).
- **Balanced classes:** 3,334 / 3,333 / 3,333 (`recommended` / `not_recommended` / `no_idea`), using the same remainder rule as the training batches.
- **Unique IDs:** comment IDs are unique and never used in train or validation.

To avoid a few popular products dominating the test set, each product contributes at most `MAX_PER_PRODUCT_PER_CLASS` comments per class.

The source file is never modified. The test products are appended to `reserved_product_ids.csv` (a backup of the previous file is kept) so that any future batch stays disjoint from the test set.


### 5.1 Settings and normalization helpers

In [6]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# --------------------------------------------------
# Test set settings
# --------------------------------------------------

TEST_SIZE = 10_000
TEST_SEED = RANDOM_SEED + 999          # separate seed, independent of validation/batches

DATA_DIR = Path(DATA_DIR)
TEST_PATH = DATA_DIR / "test_10k.csv"
RESERVED_PRODUCTS_PATH = DATA_DIR / "reserved_product_ids.csv"
RESERVED_BACKUP_PATH = DATA_DIR / "reserved_product_ids_before_test.csv"

# Everything that must never leak into the test set:
# all training batches + both validation files (original and corrected)
TRAIN_BATCH_PATTERN = "train_batch_*.csv"
VALIDATION_FILES = [
    DATA_DIR / "validation_50k.csv",
    DATA_DIR / "validation_50k_corrected.csv",
]

# Selection settings
MAX_PER_PRODUCT_PER_CLASS = 3   # a product contributes at most 3 comments per class (None = no limit)
NEAR_DUP_THRESHOLD = 0.90       # cosine similarity above which two test bodies count as "too similar"
OVERSAMPLE = 1.5                # candidates drawn per class before near-duplicate removal
MIN_KEY_LENGTH = 2              # bodies shorter than this (after normalization) are dropped
SOURCE_CHUNK = 200_000

if TEST_PATH.exists():
    raise FileExistsError(
        f"{TEST_PATH.name} already exists. Delete or rename it before re-running."
    )

# --------------------------------------------------
# Quotas (same remainder logic as the training batches)
# --------------------------------------------------

base_quota = TEST_SIZE // len(LABELS)
remainder = TEST_SIZE % len(LABELS)
TEST_QUOTAS = {
    label: base_quota + (i < remainder)
    for i, label in enumerate(LABELS)
}
print("Test quotas:", TEST_QUOTAS)

# --------------------------------------------------
# Body normalization used ONLY for duplicate detection
# (the saved text itself is never modified)
# --------------------------------------------------

_CHAR_MAP = {
    "ي": "ی", "ى": "ی", "ئ": "ی",          # Arabic yeh variants -> Persian yeh
    "ك": "ک",                               # Arabic kaf -> Persian kaf
    "ة": "ه", "ۀ": "ه",
    "أ": "ا", "إ": "ا", "آ": "ا", "ؤ": "و",
    "\u200c": " ",                          # ZWNJ -> space
    "\u200d": "", "\u200e": "", "\u200f": "",
    "\u0640": "",                           # tatweel
}
for _i, (_fa, _ar) in enumerate(zip("۰۱۲۳۴۵۶۷۸۹", "٠١٢٣٤٥٦٧٨٩")):
    _CHAR_MAP[_fa] = str(_i)
    _CHAR_MAP[_ar] = str(_i)
for _cp in list(range(0x064B, 0x0660)) + [0x0670]:   # Arabic diacritics
    _CHAR_MAP[chr(_cp)] = ""

_TRANSLATION = str.maketrans(_CHAR_MAP)


def normalize_spaced(series):
    """Lowercase, unify Arabic/Persian characters and digits, drop punctuation,
    emojis and diacritics, collapse repeated letters ("عالییییی" -> "عالی")."""
    s = series.astype(str).str.translate(_TRANSLATION).str.lower()
    s = s.str.replace(r"[\W_]+", " ", regex=True)
    s = s.str.replace(r"([^\W\d_])\1+", r"\1", regex=True)
    return s.str.strip()


def hash_strings(series):
    return pd.util.hash_pandas_object(series, index=False).to_numpy(dtype="uint64")


def compute_signatures(text_series):
    """Two duplicate signatures per body:
    - key_hash:      normalized body without any spaces (catches spacing/ZWNJ/punctuation variants)
    - tokenset_hash: sorted set of words (catches re-ordered bodies)
    """
    spaced = normalize_spaced(text_series)
    key = spaced.str.replace(" ", "", regex=False)
    token_set = spaced.map(lambda s: " ".join(sorted(set(s.split()))))
    return pd.DataFrame(
        {
            "key_len": key.str.len().to_numpy(),
            "key_hash": hash_strings(key),
            "tokenset_hash": hash_strings(token_set),
        },
        index=text_series.index,
    )


def find_near_duplicate_pairs(texts, threshold, block=2000):
    """Return index pairs (i < j) whose char n-gram cosine similarity >= threshold.
    No IDF is used, so the similarity of a pair does not depend on the rest of the corpus."""
    vec = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5), use_idf=False, sublinear_tf=True
    )
    X = vec.fit_transform(normalize_spaced(texts)).tocsr()
    pi, pj = [], []
    for start in range(0, X.shape[0], block):
        sim = (X[start:start + block] @ X.T).tocoo()
        keep = (sim.col > sim.row + start) & (sim.data >= threshold)
        pi.append(sim.row[keep] + start)
        pj.append(sim.col[keep])
    if not pi:
        return np.array([], dtype=int), np.array([], dtype=int)
    return np.concatenate(pi), np.concatenate(pj)

Test quotas: {'recommended': 3334, 'not_recommended': 3333, 'no_idea': 3333}


### 5.2 Load everything already used by train and validation

In [7]:
# --------------------------------------------------
# Load everything that is already used by train / validation
# --------------------------------------------------

def read_reference_file(path):
    """Read id, product_id and body text from a train/validation file."""
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    text_col = next((c for c in ("text", "body_normalized", "body") if c in cols), None)
    if text_col is None or "product_id" not in cols or ID_COL not in cols:
        raise ValueError(f"{path.name}: needs id, product_id and a text column. Found: {cols}")
    df = pd.read_csv(
        path,
        usecols=[ID_COL, "product_id", text_col],
        dtype=str,
        keep_default_na=False,
    ).rename(columns={text_col: TEXT_COL})
    df[ID_COL] = df[ID_COL].str.strip()
    df["product_id"] = df["product_id"].str.strip()
    return df


train_files = sorted(DATA_DIR.glob(TRAIN_BATCH_PATTERN))
validation_files = [p for p in VALIDATION_FILES if p.exists()]

if not train_files:
    raise FileNotFoundError(f"No files matching {TRAIN_BATCH_PATTERN} in {DATA_DIR}")
if not validation_files:
    raise FileNotFoundError("No validation file found.")

blocked_products = set()
blocked_ids = set()
key_hashes = {"train": [], "validation": []}
tokenset_hashes = {"train": [], "validation": []}

for group, files in (("train", train_files), ("validation", validation_files)):
    for path in files:
        ref = read_reference_file(path)
        sig = compute_signatures(ref[TEXT_COL])
        blocked_products.update(ref["product_id"])
        blocked_ids.update(ref[ID_COL])
        key_hashes[group].append(sig["key_hash"].to_numpy())
        tokenset_hashes[group].append(sig["tokenset_hash"].to_numpy())
        print(f"{path.name:35s} rows={len(ref):>9,}  products={ref['product_id'].nunique():>7,}")
        del ref, sig

# Products already recorded as reserved (validation + all batches generated so far)
if RESERVED_PRODUCTS_PATH.exists():
    reserved_file_products = set(
        pd.read_csv(RESERVED_PRODUCTS_PATH, dtype=str)["product_id"].str.strip()
    )
    blocked_products.update(reserved_file_products)
else:
    raise FileNotFoundError("reserved_product_ids.csv not found.")

train_key_hashes = np.unique(np.concatenate(key_hashes["train"]))
val_key_hashes = np.unique(np.concatenate(key_hashes["validation"]))
blocked_key_hashes = np.union1d(train_key_hashes, val_key_hashes)
blocked_tokenset_hashes = np.unique(
    np.concatenate(tokenset_hashes["train"] + tokenset_hashes["validation"])
)

print("\nBlocked products   :", f"{len(blocked_products):,}")
print("Blocked comment IDs:", f"{len(blocked_ids):,}")
print("Blocked unique bodies (normalized):", f"{len(blocked_key_hashes):,}")

# Informational: the notebook only separates train/validation by product,
# so identical bodies (e.g. "عالی") can still be shared between them.
shared_train_val = np.intersect1d(train_key_hashes, val_key_hashes).size
print(f"\n[info] Distinct bodies shared between train and validation: {shared_train_val:,}")

train_batch_01.csv                  rows=  100,000  products=  2,219
train_batch_02.csv                  rows=  100,000  products=  2,494
train_batch_03.csv                  rows=  100,000  products=  2,776
train_batch_04.csv                  rows=  100,000  products=  3,139
train_batch_05.csv                  rows=  100,000  products=  3,549
train_batch_06.csv                  rows=  100,000  products=  5,851
train_batch_07.csv                  rows=  100,000  products=  4,389
train_batch_08.csv                  rows=  100,000  products=  7,902
train_batch_09.csv                  rows=  100,000  products= 11,547
train_batch_10.csv                  rows=  100,000  products= 16,591
validation_50k.csv                  rows=   50,000  products=    999
validation_50k_corrected.csv        rows=   50,000  products=    999

Blocked products   : 61,466
Blocked comment IDs: 1,050,000
Blocked unique bodies (normalized): 963,047

[info] Distinct bodies shared between train and validation: 3,584


### 5.3 Collect eligible candidates from the source file

In [8]:
# --------------------------------------------------
# Stream the source file and collect eligible candidates
# --------------------------------------------------

source_columns = pd.read_csv(SOURCE_PATH, nrows=0).columns.tolist()
text_source_col = "text" if "text" in source_columns else "body_normalized"

required_columns = [ID_COL, "product_id", text_source_col, TARGET_COL, RATING_COL]
missing_columns = [c for c in required_columns if c not in source_columns]
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

pool_parts = []
seen_ids = set()
funnel = Counter()

for chunk in pd.read_csv(
    SOURCE_PATH,
    usecols=required_columns,
    chunksize=SOURCE_CHUNK,
    dtype=str,
    keep_default_na=False,
):
    funnel["rows_read"] += len(chunk)

    chunk[ID_COL] = chunk[ID_COL].str.strip()
    chunk["product_id"] = chunk["product_id"].str.strip()
    chunk[TARGET_COL] = chunk[TARGET_COL].str.strip().str.lower()

    # 1) valid fields
    valid = (
        chunk[ID_COL].ne("")
        & chunk["product_id"].ne("")
        & chunk[text_source_col].str.strip().ne("")
        & chunk[TARGET_COL].isin(LABELS)
    )
    funnel["dropped_invalid_fields_or_label"] += int((~valid).sum())
    chunk = chunk[valid]

    # 2) product-level separation from train + validation
    in_blocked_product = chunk["product_id"].isin(blocked_products)
    funnel["dropped_product_used_in_train_or_val"] += int(in_blocked_product.sum())
    chunk = chunk[~in_blocked_product]

    # 3) unique comment IDs (also never used in train/validation)
    bad_id = (
        chunk[ID_COL].isin(blocked_ids)
        | chunk[ID_COL].isin(seen_ids)
        | chunk[ID_COL].duplicated()
    )
    funnel["dropped_duplicate_or_used_id"] += int(bad_id.sum())
    chunk = chunk[~bad_id]
    seen_ids.update(chunk[ID_COL])

    if chunk.empty:
        continue

    # 4) body-level separation from train + validation
    chunk = chunk.rename(columns={text_source_col: TEXT_COL})
    sig = compute_signatures(chunk[TEXT_COL])
    chunk = pd.concat([chunk, sig], axis=1)

    too_short = chunk["key_len"] < MIN_KEY_LENGTH
    funnel["dropped_body_too_short_after_normalization"] += int(too_short.sum())
    chunk = chunk[~too_short]

    body_leak = (
        np.isin(chunk["key_hash"].to_numpy(), blocked_key_hashes)
        | np.isin(chunk["tokenset_hash"].to_numpy(), blocked_tokenset_hashes)
    )
    funnel["dropped_body_present_in_train_or_val"] += int(body_leak.sum())
    chunk = chunk[~body_leak]

    funnel["kept_in_pool"] += len(chunk)
    if not chunk.empty:
        pool_parts.append(chunk)

pool = pd.concat(pool_parts, ignore_index=True)
del pool_parts

print("Pool funnel:")
for name, value in funnel.items():
    print(f"  {name:48s} {value:>12,}")

print("\nPool class distribution:")
print(pool[TARGET_COL].value_counts().reindex(LABELS, fill_value=0))
print("Pool products:", f"{pool['product_id'].nunique():,}")

Pool funnel:
  rows_read                                           4,120,420
  dropped_invalid_fields_or_label                             0
  dropped_product_used_in_train_or_val                3,354,126
  dropped_duplicate_or_used_id                                0
  dropped_body_too_short_after_normalization                 28
  dropped_body_present_in_train_or_val                  117,190
  kept_in_pool                                          649,076

Pool class distribution:
recommendation_status
recommended        467056
not_recommended     94748
no_idea             87272
Name: count, dtype: int64
Pool products: 215,503


### 5.4 Clean the pool and select the test set

In [9]:
# --------------------------------------------------
# Clean the pool and select the test set
# --------------------------------------------------

def drop_label_conflicts(df, hash_col):
    """Remove every row whose body signature appears with more than one label."""
    n_labels = df.groupby(hash_col)[TARGET_COL].transform("nunique")
    return df[n_labels == 1]

pool_size_start = len(pool)

# 1) Bodies that appear with different labels are ambiguous -> remove all copies
pool = drop_label_conflicts(pool, "key_hash")
pool = drop_label_conflicts(pool, "tokenset_hash")
print("Removed (label-conflicting bodies)      :", f"{pool_size_start - len(pool):,}")

# 2) Shuffle once (seeded); the shuffled order is the priority order from here on
pool = pool.sample(frac=1, random_state=TEST_SEED).reset_index(drop=True)

# 3) Keep exactly one comment per distinct body
before = len(pool)
pool = pool.drop_duplicates("key_hash", keep="first")
pool = pool.drop_duplicates("tokenset_hash", keep="first")
print("Removed (duplicate bodies inside pool)  :", f"{before - len(pool):,}")

# 4) Limit how many comments a single product can contribute per class
if MAX_PER_PRODUCT_PER_CLASS is not None:
    before = len(pool)
    rank_in_group = pool.groupby(["product_id", TARGET_COL]).cumcount()
    pool = pool[rank_in_group < MAX_PER_PRODUCT_PER_CLASS]
    print("Removed (per-product cap)               :", f"{before - len(pool):,}")

available = pool[TARGET_COL].value_counts().reindex(LABELS, fill_value=0)
print("\nAvailable after cleaning:")
print(available)

for label in LABELS:
    if available[label] < TEST_QUOTAS[label]:
        raise ValueError(
            f"Not enough clean comments for '{label}': "
            f"available {available[label]}, required {TEST_QUOTAS[label]}. "
            "Increase MAX_PER_PRODUCT_PER_CLASS (or set it to None)."
        )

# 5) Draw a candidate set per class (a bit more than needed)
candidates = pd.concat(
    [
        pool[pool[TARGET_COL] == label].head(
            int(np.ceil(TEST_QUOTAS[label] * OVERSAMPLE))
        )
        for label in LABELS
    ]
).sample(frac=1, random_state=TEST_SEED + 1).reset_index(drop=True)

# 6) Remove near-duplicate bodies inside the candidates (greedy, in priority order)
pi, pj = find_near_duplicate_pairs(candidates[TEXT_COL], NEAR_DUP_THRESHOLD)
order = np.argsort(pi, kind="stable")
pi, pj = pi[order], pj[order]

removed = np.zeros(len(candidates), dtype=bool)
for i, j in zip(pi, pj):
    if not removed[i]:
        removed[j] = True

print(f"\nNear-duplicate pairs found: {len(pi):,} -> removed rows: {int(removed.sum()):,}")
candidates = candidates[~removed]

# 7) Take the exact quota per class
parts = []
for label in LABELS:
    class_rows = candidates[candidates[TARGET_COL] == label]
    if len(class_rows) < TEST_QUOTAS[label]:
        raise ValueError(
            f"Not enough candidates for '{label}' after near-duplicate removal "
            f"({len(class_rows)} < {TEST_QUOTAS[label]}). Increase OVERSAMPLE."
        )
    parts.append(class_rows.head(TEST_QUOTAS[label]))

test_df = (
    pd.concat(parts)
    .sample(frac=1, random_state=TEST_SEED + 2)
    .reset_index(drop=True)
)

test_df = test_df[[ID_COL, "product_id", TEXT_COL, TARGET_COL, RATING_COL]].copy()
test_df[RATING_COL] = pd.to_numeric(test_df[RATING_COL], errors="coerce")

print("\nTest shape:", test_df.shape)
print(test_df[TARGET_COL].value_counts().reindex(LABELS))

Removed (label-conflicting bodies)      : 1,005
Removed (duplicate bodies inside pool)  : 21,754
Removed (per-product cap)               : 119,742

Available after cleaning:
recommendation_status
recommended        330576
not_recommended     90733
no_idea             85266
Name: count, dtype: int64

Near-duplicate pairs found: 7 -> removed rows: 7

Test shape: (10000, 5)
recommendation_status
recommended        3334
not_recommended    3333
no_idea            3333
Name: count, dtype: int64


### 5.5 Verify, save and reserve the test products

In [10]:
# --------------------------------------------------
# Verification: every guarantee is checked explicitly
# --------------------------------------------------

def verify_test_set(df, label=""):
    df = df.copy()
    df[ID_COL] = df[ID_COL].astype(str).str.strip()
    df["product_id"] = df["product_id"].astype(str).str.strip()
    df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()

    # Size and balance
    assert len(df) == TEST_SIZE, f"Wrong size: {len(df)}"
    counts = df[TARGET_COL].value_counts().reindex(LABELS, fill_value=0).to_dict()
    assert counts == TEST_QUOTAS, f"Class counts differ from quotas: {counts}"

    # Valid fields
    assert df[TEXT_COL].astype(str).str.strip().ne("").all(), "Empty body found"
    assert df[TARGET_COL].isin(LABELS).all(), "Invalid label found"
    assert df[ID_COL].is_unique, "Duplicate comment IDs inside test"
    assert not df[ID_COL].isin(blocked_ids).any(), "Comment ID shared with train/validation"

    # Product-level leakage
    shared_products = set(df["product_id"]) & blocked_products
    assert not shared_products, f"{len(shared_products)} products shared with train/validation"

    # Body-level leakage against train + validation
    sig = compute_signatures(df[TEXT_COL])
    assert (sig["key_len"] >= MIN_KEY_LENGTH).all(), "Too-short body found"
    assert not np.isin(sig["key_hash"].to_numpy(), train_key_hashes).any(), "Body shared with train"
    assert not np.isin(sig["key_hash"].to_numpy(), val_key_hashes).any(), "Body shared with validation"
    assert not np.isin(sig["tokenset_hash"].to_numpy(), blocked_tokenset_hashes).any(), \
        "Re-ordered body shared with train/validation"

    # Duplicates inside the test set
    assert sig["key_hash"].is_unique, "Duplicate bodies inside test"
    assert sig["tokenset_hash"].is_unique, "Re-ordered duplicate bodies inside test"
    pi, _ = find_near_duplicate_pairs(df[TEXT_COL], NEAR_DUP_THRESHOLD)
    assert len(pi) == 0, f"{len(pi)} near-duplicate pairs inside test"

    print(f"[{label}] all checks passed")


verify_test_set(test_df, label="in memory")

# Save, then re-load from disk and verify the file itself
test_df.to_csv(TEST_PATH, index=False, encoding="utf-8-sig")

reloaded = pd.read_csv(TEST_PATH, dtype=str, keep_default_na=False)
assert list(reloaded.columns) == [ID_COL, "product_id", TEXT_COL, TARGET_COL, RATING_COL]
verify_test_set(reloaded, label="saved file")

# --------------------------------------------------
# Record the test products as reserved (with a backup of the old file)
# --------------------------------------------------

if not RESERVED_BACKUP_PATH.exists():
    pd.read_csv(RESERVED_PRODUCTS_PATH, dtype=str).to_csv(
        RESERVED_BACKUP_PATH, index=False, encoding="utf-8-sig"
    )

test_product_ids = set(test_df["product_id"].astype(str).str.strip())
all_reserved = blocked_products | test_product_ids

pd.DataFrame({"product_id": sorted(all_reserved)}).to_csv(
    RESERVED_PRODUCTS_PATH, index=False, encoding="utf-8-sig"
)

# --------------------------------------------------
# Summary
# --------------------------------------------------

print("\nSaved:", TEST_PATH.name)
print("Test shape:", test_df.shape)

print("\nClass distribution:")
print(test_df[TARGET_COL].value_counts().reindex(LABELS))

print("\nUnique comment IDs :", test_df[ID_COL].nunique())
print("Unique products    :", test_df["product_id"].nunique())
print("Max comments per product:", test_df["product_id"].value_counts().max())
print("Unique normalized bodies:", compute_signatures(test_df[TEXT_COL])["key_hash"].nunique())

print("\nBody length (characters) per class:")
print(test_df.groupby(TARGET_COL)[TEXT_COL].apply(lambda s: s.str.len()).groupby(level=0).describe()[["mean", "50%", "max"]])

print("\nRate distribution per class:")
print(pd.crosstab(test_df[TARGET_COL], test_df[RATING_COL], dropna=False))

print("\nReserved products (train + validation + test):", f"{len(all_reserved):,}")

[in memory] all checks passed
[saved file] all checks passed

Saved: test_10k.csv
Test shape: (10000, 5)

Class distribution:
recommendation_status
recommended        3334
not_recommended    3333
no_idea            3333
Name: count, dtype: int64

Unique comment IDs : 10000
Unique products    : 9694
Max comments per product: 3
Unique normalized bodies: 10000

Body length (characters) per class:
                            mean   50%     max
recommendation_status                         
no_idea                80.531053  54.0  1569.0
not_recommended        88.926793  63.0  1716.0
recommended            63.115477  43.0  2618.0

Rate distribution per class:
rate                   0.00  0.05  0.20  0.25  0.50  0.60  0.75  0.80  0.90  \
recommendation_status                                                         
no_idea                 146     0     0     0     0     0     1     0     0   
not_recommended         376     1     1     2     3     2     1     1     1   
recommended           

In [11]:
test = pd.read_csv("../data/test_10k.csv")
test.head(20)

,id,product_id,text,recommendation_status,rate
0,9347051,622271,چیزی ک تو عکس دیدم تا چیزی ک بدستم رسید اصلآ ی...,not_recommended,0.00
1,33634187,37649,جنس کاعذ را جدیدا از نوع بالکی یا کاهی بکار می...,recommended,0.00
2,38638350,4526503,فکر می کردم ساخت آلمان باشه ولی مداد فیک هست و...,no_idea,2.00
3,53441720,3286282,بسیار ناز و بی کیفیت قیمت بالا سایز کوچکتر از ...,not_recommended,2.00
4,54362934,11135856,کیفیت معمولی و نیازمند استفاده از بهبود دهنده,no_idea,3.00
5,9870701,448138,طراحی زیبا و جنس خوبی داشت اما قیمتش یه کم بال...,recommended,3.40
6,41883585,9276955,اندازه عروسک تحویلی با توی سایت فرق داشت 20 سا...,no_idea,4.00
7,30529192,4837287,خوبه این رنگ رو چون کم داشتم تک خریدمش,recommended,3.00
8,47955314,4513749,صابون اشتباه ارسال شد بجای آووکادو آلوئه ورا ا...,no_idea,1.00
9,4762729,1882028,ضخامتش صدای خیلی زیرتری نسبت به ناخن داره,no_idea,3.60
